# Chapter 4: cut a trace into events

Section 4.4 cuts one synthetic fermenter trace into fill, transfer and CIP events, then prints the
unmatched ones. The trace is FV07: two days around the fill of F07-26-034 in March and the transfer and CIP
in April.

In [ ]:
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
import pandas as pd
import kestrel as k

readings = k.load("tag_reading")
trace = k.wide_trace(readings, "FV07")
print(trace.describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (start, end) in zip(axes, [("2026-03-12", "2026-03-12 23:59"), ("2026-04-10 00:30", "2026-04-11 12:00")]):
    part = trace.loc[start:end]
    ax.plot(part.index, part.level, label="level hL")
    ax.plot(part.index, part.temp * 10, label="temperature x 10")
    ax.tick_params(axis="x", rotation=45)
axes[0].legend()
plt.tight_layout()
plt.show()

The rule (Kestrel working values): a fill starts when the inlet valve is open and the level rises by at least
5 hL in 10 minutes. It ends when the valve closes and the level holds for 3 minutes. A transfer is the mirror
on the outlet valve. A CIP shows as a jump in return line conductivity.

In [ ]:
events = k.detect_events(trace, "FV07")
matched = k.match_events(events)
print(matched.to_string(index=False))

Unmatched events go to a short queue for a brewer to settle. Do not guess.

In [ ]:
queue = matched[matched.matched_to.isna()]
print(queue[["vessel", "event_type", "start_utc", "end_utc", "volume_hl"]].to_string(index=False))

The second transfer is the partial transfer of section 4.6: it stopped for 20 minutes and restarted, so the
rule read it as two. The transfer log holds one.